# Диагностика "грязи" в данных (items_human.parquet)

Цель: понять, что нужно почистить перед построением эмбеддингов (deepvk/USER-bge-m3) для матчинга.

In [1]:
import polars as pl
import json
from collections import Counter

df = pl.read_parquet('data/items_human.parquet')
print(df.shape)
df.head()

(711304, 4)


id,name,attributes,category
i64,str,str,str
197,"""victor reinz п…","""{""артикул"":""70…","""Автотовары"""
415,"""stellox диск т…","""{""артикул"":""st…","""Автотовары"""
427,"""комплект подши…","""{""артикул прои…","""Автотовары"""
1027,"""kraft подшипни…","""{""артикул"":""11…","""Автотовары"""
2639,"""фильтр салонны…","""{""альтернативн…","""Автотовары"""


## 1. Дубликаты

In [2]:
print('дубли id:', df.height - df['id'].n_unique())
print('полностью одинаковые строки:', df.height - df.unique().height)
print('дубли (name, category):', df.height - df.select(['name', 'category']).n_unique())

df.group_by(['name', 'category']).len().filter(pl.col('len') > 1).sort('len', descending=True).head(10)

дубли id: 0
полностью одинаковые строки: 0
дубли (name, category): 97400


name,category,len
str,str,u32
"""кроссовки nike…","""Обувь""",841
"""кроссовки new …","""Обувь""",540
"""кроссовки adid…","""Обувь""",369
"""футболка""","""Одежда""",357
"""золото дисконт…","""Ювелирные изде…",331
"""кроссовки asic…","""Обувь""",320
"""avtolider1 опл…","""Автотовары""",262
"""casio часы нар…","""Галантерея и а…",223
"""золото дисконт…","""Ювелирные изде…",215


## 2. Качество текста в `name`

In [3]:
name_len = pl.col('name').str.len_chars()

print('длина name: min', df.select(name_len.min()).item(), 'max', df.select(name_len.max()).item(),
      'mean', round(df.select(name_len.mean()).item(), 1))

print('очень короткие (<5 символов):', df.filter(name_len < 5).height)
print('очень длинные (>500 символов):', df.filter(name_len > 500).height)
print('двойные пробелы:', df.filter(pl.col('name').str.contains('  ')).height)
print('пробелы по краям:', df.filter(pl.col('name') != pl.col('name').str.strip_chars()).height)
print('неразрывный пробел (nbsp):', df.filter(pl.col('name').str.contains('\u00a0')).height)
print('переносы строк:', df.filter(pl.col('name').str.contains('\n')).height)
print('html-теги:', df.filter(pl.col('name').str.contains('<[a-zA-Z/][^>]*>')).height)
print('html-entities:', df.filter(pl.col('name').str.contains('&[a-z]+;|&#[0-9]+;')).height)

длина name: min 1 max 4126 mean 57.7
очень короткие (<5 символов): 203
очень длинные (>500 символов): 1
двойные пробелы: 8086
пробелы по краям: 6661
неразрывный пробел (nbsp): 1062
переносы строк: 44
html-теги: 3
html-entities: 172


In [4]:
print('=== примеры очень коротких названий ===')
print(df.filter(name_len < 5).select('name', 'category').head(15))

print('\n=== примеры очень длинных названий (спам/перечисление партномеров) ===')
for row in df.filter(name_len > 500).head(3).iter_rows(named=True):
    print(row['category'], '|', row['name'][:200], '...')

=== примеры очень коротких названий ===
shape: (15, 2)
┌──────┬─────────────────┐
│ name ┆ category        │
│ ---  ┆ ---             │
│ str  ┆ str             │
╞══════╪═════════════════╡
│ болт ┆ Автотовары      │
│ шрус ┆ Автотовары      │
│ очки ┆ Аптека          │
│ очки ┆ Аптека          │
│ очки ┆ Аптека          │
│ …    ┆ …               │
│ /    ┆ Бытовая техника │
│ утюг ┆ Бытовая техника │
│ фен  ┆ Бытовая техника │
│ фены ┆ Бытовая техника │
│ утюг ┆ Бытовая техника │
└──────┴─────────────────┘

=== примеры очень длинных названий (спам/перечисление партномеров) ===
Электроника | iqzip аккумулятор для ноутбука, (совместимые партномера:
  pa3533u-1bas, pa3533u-1brs, pa3534u-1bas, pa3534u-1brs, pa3535u-1bas, pa3535u-1brs, pa3682u-1brs, pa3727u-1brs, pabas097, pabas098, pabas099, ...


## 3. Качество `attributes`

In [5]:
print('пустые attributes ({}):', df.filter(pl.col('attributes').str.strip_chars() == '{}').height)

key_counter = Counter()
dirty_value_rows = 0
for s in df['attributes']:
    d = json.loads(s)
    key_counter.update(d.keys())
    if any(isinstance(v, str) and (v != v.strip() or '  ' in v) for v in d.values()):
        dirty_value_rows += 1

print('всего уникальных ключей атрибутов:', len(key_counter))
print('ключи, встречающиеся <=2 раз (мусорный длинный хвост):',
      sum(1 for v in key_counter.values() if v <= 2))
print('строк с "грязными" пробелами в значениях атрибутов:', dirty_value_rows)

пустые attributes ({}): 72


всего уникальных ключей атрибутов: 34112
ключи, встречающиеся <=2 раз (мусорный длинный хвост): 12506
строк с "грязными" пробелами в значениях атрибутов: 22501


In [6]:
# ключи, различающиеся только регистром - потенциальные дубли-синонимы
lower_map = {}
for k in key_counter:
    lower_map.setdefault(k.lower(), []).append(k)

case_variants = {k: v for k, v in lower_map.items() if len(v) > 1}
print('групп ключей-регистровых дублей:', len(case_variants))

print('\nсамые редкие ключи (пример мусора):')
for k, v in key_counter.most_common()[-15:]:
    print(' ', v, k)

групп ключей-регистровых дублей: 0

самые редкие ключи (пример мусора):
  1 края
  1 нескользящая поверхность
  1 рекомендуемая ширина шва (мм)
  1 упаковка (м²)
  1 количество плиток в упаковке
  1 тнвэд
  1 назначение подарка
  1 вид броши
  1 немагнитная сталь
  1 музыкальные жанры
  1 формат релиза
  1 толщина стержня
  1 проба серебра
  1 форма ионизатора
  1 с тонкой ручкой


## 4. Итоговая сводка

In [7]:
total = df.height
print(f"Всего товаров: {total}")
print(f"Дубли (name, category): {total - df.select(['name', 'category']).n_unique()} ({(total - df.select(['name', 'category']).n_unique())/total:.1%})")
print(f"Названия <5 символов: {df.filter(name_len < 5).height} ({df.filter(name_len < 5).height/total:.1%})")
print(f"Названия >500 символов: {df.filter(name_len > 500).height} ({df.filter(name_len > 500).height/total:.1%})")
print(f"Двойные пробелы в названии: {df.filter(pl.col('name').str.contains('  ')).height}")
print(f"Пробелы по краям: {df.filter(pl.col('name') != pl.col('name').str.strip_chars()).height}")
print(f"Пустые attributes: {df.filter(pl.col('attributes').str.strip_chars() == '{}').height}")
print(f"Уникальных ключей атрибутов: {len(key_counter)} (из них <=2 упоминаний: {sum(1 for v in key_counter.values() if v <= 2)})")

Всего товаров: 711304
Дубли (name, category): 97400 (13.7%)
Названия <5 символов: 203 (0.0%)
Названия >500 символов: 1 (0.0%)
Двойные пробелы в названии: 8086
Пробелы по краям: 6661


Пустые attributes: 72
Уникальных ключей атрибутов: 34112 (из них <=2 упоминаний: 12506)
